In [ ]:
!pip install -q kagglehub

import kagglehub
import os

new_path = kagglehub.dataset_download(
    "dorianlazar/medium-articles-dataset"
)

print("Downloaded files:")

for root, dirs, files in os.walk(new_path):
    for file in files:
        print(os.path.join(root, file))

100%|██████████| 1.33G/1.33G [00:18<00:00, 78.2MB/s]

Extracting files...


Downloaded files:
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/medium_data.csv
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/1045.jpeg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/6045.png
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/6482.jpg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/2977.jpeg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/120.jpeg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/1123.jpg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/1296.jpeg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/808.jpeg
/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/images/214.png
/root/.cache/kagglehub/dataset

In [ ]:
import pandas as pd
import os
import glob

csv_files = glob.glob(
    os.path.join(new_path, "**", "*.csv"),
    recursive=True
)

print("CSV files found:", csv_files)

df = pd.read_csv(csv_files[0])

print("\nDataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 3 rows:")
display(df.head(3))

CSV files found: ['/root/.cache/kagglehub/datasets/dorianlazar/medium-articles-dataset/versions/1/medium_data.csv']

Dataset shape: (6508, 10)

Column names: ['id', 'url', 'title', 'subtitle', 'image', 'claps', 'responses', 'reading_time', 'publication', 'date']

First 3 rows:


,id,url,title,subtitle,image,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/a-beginners-gui...,A Beginner’s Guide to Word Embedding with Gens...,NaN,1.png,850,8,8,Towards Data Science,2019-05-30
1,2,https://towardsdatascience.com/hands-on-graph-...,Hands-on Graph Neural Networks with PyTorch & ...,NaN,2.png,1100,11,9,Towards Data Science,2019-05-30
2,3,https://towardsdatascience.com/how-to-use-ggpl...,How to Use ggplot2 in Python,A Grammar of Graphics for Python,3.png,767,1,5,Towards Data Science,2019-05-30


In [ ]:

import pandas as pd
import re

# Use article titles and subtitles
df = df.fillna("")

df["text"] = df["title"].astype(str) + " " + df["subtitle"].astype(str)

# Clean the text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

# Remove empty and very short texts
sentences = [
    text for text in df["text"].tolist()
    if len(text.split()) >= 3
]

print("Total articles:", len(df))
print("Usable text samples:", len(sentences))
print("\nFirst 10 samples:")
for sentence in sentences[:10]:
    print(sentence)


Total articles: 6508
Usable text samples: 6468

First 10 samples:
a beginner s guide to word embedding with gensim word2vec model
hands on graph neural networks with pytorch pytorch geometric
how to use ggplot2 in python a grammar of graphics for python
databricks how to save files in csv on your local computer when i work on python projects dealing
a step by step implementation of gradient descent and backpropagation one example of building neural
an easy introduction to sql for data scientists
hypothesis testing visualized literally seeing how stat tests work
introduction to latent matrix factorization recommender systems
which 2020 candidate is the best at twitter a data analysis of the 2020 democratic candidate twitter accounts
what if ai model understanding were easy see analytics for ai in action in this what if tool


In [ ]:

from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer(
    oov_token="<OOV>",
    filters=""
)

tokenizer.fit_on_texts(sentences)

# Convert words into numerical sequences
sequences = tokenizer.texts_to_sequences(sentences)

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

print("Total text samples:", len(sentences))
print("Vocabulary size:", vocab_size)
print("\nFirst sentence:")
print(sentences[0])

print("\nTokenized sequence:")
print(sequences[0])

print("\nFirst 15 words in vocabulary:")
print(list(tokenizer.word_index.items())[:15])


Total text samples: 6468
Vocabulary size: 9491

First sentence:
a beginner s guide to word embedding with gensim word2vec model

Tokenized sequence:
[4, 589, 15, 66, 3, 384, 1777, 16, 2526, 3260, 118]

First 15 words in vocabulary:
[('<OOV>', 1), ('the', 2), ('to', 3), ('a', 4), ('and', 5), ('of', 6), ('strong', 7), ('how', 8), ('you', 9), ('your', 10), ('in', 11), ('for', 12), ('markup', 13), ('is', 14), ('s', 15)]


In [ ]:

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Generate input sequences
input_sequences = []

for sequence in sequences:
    for i in range(2, len(sequence)):
        input_sequences.append(sequence[:i+1])

# Make all sequences the same length
max_sequence_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre"
)

# Separate input words and target word
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Total training examples:", len(X))
print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Maximum sequence length:", max_sequence_len)

print("\nExample input:", X[0])
print("Example target word ID:", y[0])


Total training examples: 72446
Input shape: (72446, 44)
Target shape: (72446,)
Maximum sequence length: 45

Example input: [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   4 589]
Example target word ID: 15


In [ ]:
import os

csv_path = csv_files[0]

size_mb = os.path.getsize(csv_path) / (1024 * 1024)

print(f"CSV file size: {size_mb:.2f} MB")

CSV file size: 1.36 MB


In [ ]:

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Split sentences first
train_sentences, test_sentences = train_test_split(
    sentences,
    test_size=0.20,
    random_state=42
)

# Create input-target sequences separately
def create_sequences(texts, tokenizer):
    result = []
    for text in texts:
        tokens = tokenizer.texts_to_sequences([text])[0]
        for i in range(2, len(tokens)):
            result.append(tokens[:i+1])
    return result

train_sequences = create_sequences(train_sentences, tokenizer)
test_sequences = create_sequences(test_sentences, tokenizer)

max_sequence_len = max(len(s) for s in train_sequences)

train_sequences = pad_sequences(
    train_sequences, maxlen=max_sequence_len, padding="pre"
)
test_sequences = pad_sequences(
    test_sequences, maxlen=max_sequence_len, padding="pre"
)

X_train = train_sequences[:, :-1]
y_train = train_sequences[:, -1]

X_test = test_sequences[:, :-1]
y_test = test_sequences[:, -1]

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Vocabulary size:", vocab_size)


Training sentences: 5174
Testing sentences: 1294
X_train shape: (57897, 44)
X_test shape: (14549, 44)
Vocabulary size: 9491


In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.keras.utils.set_random_seed(42)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.build(input_shape=(None, X_train.shape[1]))

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 44, 64)         │       607,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9491)           │     1,224,339 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,930,579 (7.36 MB)

 Trainable params: 1,930,579 (7.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 97s 231ms/step - accuracy: 0.0346 - loss: 7.2763 - val_accuracy: 0.0454 - val_loss: 6.9604
Epoch 2/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 93s 229ms/step - accuracy: 0.0572 - loss: 6.7174 - val_accuracy: 0.0575 - val_loss: 6.9180
Epoch 3/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 92s 225ms/step - accuracy: 0.0774 - loss: 6.5351 - val_accuracy: 0.0819 - val_loss: 6.8576
Epoch 4/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 142s 227ms/step - accuracy: 0.0958 - loss: 6.3382 - val_accuracy: 0.0933 - val_loss: 6.7856
Epoch 5/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 141s 225ms/step - accuracy: 0.1123 - loss: 6.1451 - val_accuracy: 0.1047 - val_loss: 6.7451
Epoch 6/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 89s 219ms/step - accuracy: 0.1244 - loss: 5.9701 - val_accuracy: 0.1133 - val_loss: 6.7287
Epoch 7/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 145s 225ms/step - accuracy: 0.1334 - loss: 5.8160 - val_accuracy: 0.1200 - val_loss: 6.7326
Epoch 8/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 94s 230ms/step - accuracy: 0.1395 - loss

In [ ]:
print("Completed epochs:", len(history.epoch))
print("Epochs completed:", history.epoch)

Completed epochs: 8
Epochs completed: [0, 1, 2, 3, 4, 5, 6, 7]


In [ ]:

history_more = model.fit(
    X_train,
    y_train,
    initial_epoch=8,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("Additional epochs completed:", len(history_more.epoch))


Epoch 9/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 98s 240ms/step - accuracy: 0.1322 - loss: 5.8054 - val_accuracy: 0.1197 - val_loss: 6.7516
Epoch 10/10
408/408 ━━━━━━━━━━━━━━━━━━━━ 90s 221ms/step - accuracy: 0.1395 - loss: 5.6688 - val_accuracy: 0.1231 - val_loss: 6.7765
Additional epochs completed: 2


In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

top1 = tf.keras.metrics.SparseCategoricalAccuracy()
top5 = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)

for i in range(0, len(X_test), 128):
    xb = X_test[i:i+128]
    yb = y_test[i:i+128]
    probs = model(xb, training=False)
    top1.update_state(yb, probs)
    top5.update_state(yb, probs)

print("Top-1 Accuracy:", round(float(top1.result())*100, 2), "%")
print("Top-5 Accuracy:", round(float(top5.result())*100, 2), "%")

def predict_next_word(phrase):
    tokens = tokenizer.texts_to_sequences([phrase.lower()])[0]
    if not tokens:
        return "No known words found"

    x = pad_sequences([tokens], maxlen=X_train.shape[1], padding="pre")
    probs = model(x, training=False).numpy()[0]
    word_id = int(np.argmax(probs))
    return tokenizer.index_word.get(word_id, "<unknown>")

phrases = [
    "machine learning",
    "deep learning",
    "data science",
    "neural networks",
    "artificial intelligence"
]

print("\nFive Next-Word Predictions:")
for phrase in phrases:
    print(phrase, "->", predict_next_word(phrase))


Top-1 Accuracy: 12.58 %
Top-5 Accuracy: 24.14 %

Five Next-Word Predictions:
machine learning -> and
deep learning -> and
data science -> with
neural networks -> of
artificial intelligence -> of


In [ ]:
import tensorflow as tf

top1 = tf.keras.metrics.SparseCategoricalAccuracy()
top5 = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)

for i in range(0, len(X_test), 128):
    xb = X_test[i:i+128]
    yb = y_test[i:i+128]

    probs = model(xb, training=False)

    top1.update_state(yb, probs)
    top5.update_state(yb, probs)

print("Top-1 Accuracy:", round(float(top1.result()) * 100, 2), "%")
print("Top-5 Accuracy:", round(float(top5.result()) * 100, 2), "%")

Top-1 Accuracy: 12.58 %
Top-5 Accuracy: 24.14 %
